# Summer Olympics 2028 Medal Prediction 

In [23]:
# Load the data
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("./data")
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(exist_ok=True)

# Update filenames to your actual names
MEDALS_PATH   = DATA_DIR / "/kaggle/input/datasets/reenamaritapinto/summer-olympics-2028/olympic_medals.csv"
ATHLETES_PATH = DATA_DIR / "/kaggle/input/datasets/reenamaritapinto/summer-olympics-2028/athlete_events.csv"
POP_PATH      = DATA_DIR / "/kaggle/input/datasets/reenamaritapinto/summer-olympics-2028/population_by_country_1800_2100_gapminder_un.csv.csv"
GDPPC_PATH    = DATA_DIR / "/kaggle/input/datasets/reenamaritapinto/country-wise-per-capita/gdp-per-capita-worldbank.csv"

olympic_medals = pd.read_csv(MEDALS_PATH)
athlete_events = pd.read_csv(ATHLETES_PATH)
pop_raw        = pd.read_csv(POP_PATH)
gdp_raw        = pd.read_csv(GDPPC_PATH)

print("olympic_medals:", olympic_medals.shape)
print("athlete_events:", athlete_events.shape)
print("population:", pop_raw.shape)
print("gdp:", gdp_raw.shape)

olympic_medals: (21477, 12)
athlete_events: (271116, 15)
population: (59297, 4)
gdp: (7240, 5)


In [11]:
med = olympic_medals.copy()
med.columns = [c.strip() for c in med.columns]

# Filter Summer
med = med[med["Olympic_season"].astype(str).str.lower().eq("summer")].copy()

# Keep only "Country" rows
med = med[med["Committee_type"].astype(str).str.lower().eq("country")].copy()

# Keep medal rows only (should already be medals, but safe)
med = med[med["Medal_type"].notna()].copy()

# Standardize names for modeling
med = med.rename(columns={
    "Olympic_year": "Year",
    "Committee": "Country",
    "Code": "NOC",
    "Discipline": "Sport",
    "Medal_type": "Medal"
})

med["Year"] = pd.to_numeric(med["Year"], errors="coerce").astype("Int64")
med = med.dropna(subset=["Year", "Country", "NOC", "Sport", "Medal"]).copy()
med["Year"] = med["Year"].astype(int)

# Clean strings
for c in ["Country","NOC","Sport","Medal"]:
    med[c] = med[c].astype(str).str.strip()

print("Clean medals:", med.shape)
med.head()

Clean medals: (15896, 12)


,Olympiad,Sport,Event,Winner,Medal,Olympic_city,Year,Olympic_season,Gender,NOC,Country,Committee_type
0,Athina 1896,Artistic Gymnastics,"Horizontal Bar, Men",Hermann Weingärtner,Gold,Athens,1896,summer,Men,GER,Germany,Country
1,Athina 1896,Artistic Gymnastics,"Horizontal Bar, Men",Alfred Flatow,Silver,Athens,1896,summer,Men,GER,Germany,Country
2,Athina 1896,Artistic Gymnastics,"Horizontal Bar, Teams, Men",Germany,Gold,Athens,1896,summer,Men,GER,Germany,Country
3,Athina 1896,Artistic Gymnastics,"Horse Vault, Men",Hermann Weingärtner,Bronze,Athens,1896,summer,Men,GER,Germany,Country
4,Athina 1896,Artistic Gymnastics,"Horse Vault, Men",Carl Schuhmann,Gold,Athens,1896,summer,Men,GER,Germany,Country


In [12]:
#2) Convert medals into Year × Sport × Country counts

sport_country_year = (
    med
    .groupby(["Year","Sport","Country","NOC","Medal"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure columns exist
for col in ["Gold","Silver","Bronze"]:
    if col not in sport_country_year.columns:
        sport_country_year[col] = 0

sport_country_year["Total_Medals"] = (
    sport_country_year["Gold"] + sport_country_year["Silver"] + sport_country_year["Bronze"]
)

# Leakage-free previous medals (previous Olympics for same NOC & Sport)
sport_country_year = sport_country_year.sort_values(["NOC","Sport","Year"])
sport_country_year["Prev_Sport_Medals"] = (
    sport_country_year.groupby(["NOC","Sport"])["Total_Medals"].shift(1).fillna(0)
)

sport_country_year.head()

Medal,Year,Sport,Country,NOC,Bronze,Gold,Silver,Total_Medals,Prev_Sport_Medals
4508,2008,Taekwondo,Afghanistan,AFG,1,0,0,1,0.0
4929,2012,Taekwondo,Afghanistan,AFG,1,0,0,1,1.0
6406,2024,Wrestling,Albania,ALB,2,0,0,2,0.0
5960,2024,Artistic Gymnastics,Algeria,ALG,0,1,0,1,0.0
2752,1992,Athletics,Algeria,ALG,0,1,0,1,0.0


In [13]:
#Clean athlete_events and create Athlete_Count (proxy)
ath = athlete_events.copy()
ath.columns = [c.strip() for c in ath.columns]

ath = ath[ath["Season"].astype(str).str.lower().eq("summer")].copy()
ath["Year"] = pd.to_numeric(ath["Year"], errors="coerce").astype("Int64")
ath = ath.dropna(subset=["Year","Sport","NOC","ID"]).copy()
ath["Year"] = ath["Year"].astype(int)

athlete_count = (
    ath.groupby(["Year","Sport","NOC"])["ID"]
      .nunique()
      .reset_index(name="Athlete_Count")
)

# Merge into medal table
df = sport_country_year.merge(athlete_count, on=["Year","Sport","NOC"], how="left")

# Forward-fill Athlete_Count for missing years (common if athlete data ends earlier)
df = df.sort_values(["NOC","Sport","Year"])
df["Athlete_Count"] = (
    df.groupby(["NOC","Sport"])["Athlete_Count"].ffill().fillna(0).astype(int)
)

df.head()

,Year,Sport,Country,NOC,Bronze,Gold,Silver,Total_Medals,Prev_Sport_Medals,Athlete_Count
0,2008,Taekwondo,Afghanistan,AFG,1,0,0,1,0.0,2
1,2012,Taekwondo,Afghanistan,AFG,1,0,0,1,1.0,2
2,2024,Wrestling,Albania,ALB,2,0,0,2,0.0,0
3,2024,Artistic Gymnastics,Algeria,ALG,0,1,0,1,0.0,0
4,1992,Athletics,Algeria,ALG,0,1,0,1,0.0,9


In [14]:
# 4) Clean Population

pop = pop_raw.copy()
pop.columns = [c.strip() for c in pop.columns]

# Identify population numeric column (exclude Entity/Year)
pop_value_cols = [c for c in pop.columns if c not in ["Entity","Year"]]
if len(pop_value_cols) == 0:
    raise ValueError("Population file: no value column found besides Entity/Year.")

# pick first value column
pop_value = pop_value_cols[0]

pop_clean = pop.rename(columns={"Entity":"Country", pop_value:"Population"}).copy()
pop_clean["Year"] = pd.to_numeric(pop_clean["Year"], errors="coerce").astype("Int64")
pop_clean["Population"] = pd.to_numeric(pop_clean["Population"], errors="coerce")
pop_clean = pop_clean.dropna(subset=["Country","Year","Population"]).copy()
pop_clean["Year"] = pop_clean["Year"].astype(int)
pop_clean["Country"] = pop_clean["Country"].astype(str).str.strip()

print("pop_clean:", pop_clean.shape)
pop_clean.head()

pop_clean: (59297, 4)


,Country,Year,Population,"Population by country, 1800 to 2015 (Gapminder & UN)"
0,Afghanistan,1800,3280000,3280000.0
1,Afghanistan,1801,3280000,3280000.0
2,Afghanistan,1802,3280000,3280000.0
3,Afghanistan,1803,3280000,3280000.0
4,Afghanistan,1804,3280000,3280000.0


In [15]:
#5) Clean GDP per capita
gdp = gdp_raw.copy()
gdp.columns = [c.strip() for c in gdp.columns]

# Find the GDP per capita column (exact name from your screenshot)
gdp_pc_col = None
for c in gdp.columns:
    if "gdp per capita" in c.lower():
        gdp_pc_col = c
        break
if gdp_pc_col is None:
    raise ValueError(f"Could not find GDP per capita column. Columns: {list(gdp.columns)}")

gdp_clean = gdp.rename(columns={
    "Entity":"Country",
    "Code":"NOC",
    gdp_pc_col:"GDP_per_capita"
}).copy()

gdp_clean["Year"] = pd.to_numeric(gdp_clean["Year"], errors="coerce").astype("Int64")
gdp_clean["GDP_per_capita"] = pd.to_numeric(gdp_clean["GDP_per_capita"], errors="coerce")
gdp_clean = gdp_clean.dropna(subset=["Country","Year","GDP_per_capita"]).copy()
gdp_clean["Year"] = gdp_clean["Year"].astype(int)

for c in ["Country","NOC"]:
    gdp_clean[c] = gdp_clean[c].astype(str).str.strip()

print("gdp_clean:", gdp_clean.shape)
gdp_clean.head()

gdp_clean: (7240, 5)


,Country,NOC,Year,GDP_per_capita,World region according to OWID
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia


In [16]:
#Add Host flag (USA hosts 2028)
TARGET_YEAR = 2028

df["Is_Host"] = 0
df.loc[(df["Year"] == TARGET_YEAR) & (df["Country"].str.lower() == "united states"), "Is_Host"] = 1

In [17]:
#7) Merge everything → Final model table
model_df = df.merge(
    pop_clean[["Country","Year","Population"]],
    on=["Country","Year"],
    how="left"
).merge(
    gdp_clean[["NOC","Year","GDP_per_capita"]],
    on=["NOC","Year"],
    how="left"
)

# Fill forward macro vars (helps Olympic-year sparsity)
model_df = model_df.sort_values(["Country","Year"])
model_df["Population"] = model_df.groupby("Country")["Population"].ffill()
model_df["GDP_per_capita"] = model_df.groupby("NOC")["GDP_per_capita"].ffill()

# Restrict years based on macro availability (you said GDPpc starts 1900)
model_df = model_df[model_df["Year"] >= 1900].copy()

# Log features (recommended)
model_df["log_GDPpc"] = np.log1p(model_df["GDP_per_capita"].clip(lower=0))
model_df["log_Pop"]   = np.log1p(model_df["Population"].clip(lower=0))

FINAL_COLS = [
    "Year","Sport","Country","NOC",
    "Prev_Sport_Medals","Athlete_Count","Is_Host",
    "Population","GDP_per_capita","log_Pop","log_GDPpc",
    "Gold","Silver","Bronze","Total_Medals"
]
final_model_table = model_df[FINAL_COLS].copy()

print("final_model_table:", final_model_table.shape)
print("Missing Population %:", round(final_model_table["Population"].isna().mean()*100, 2))
print("Missing GDPpc %:", round(final_model_table["GDP_per_capita"].isna().mean()*100, 2))

final_model_table.head()

final_model_table: (6398, 15)
Missing Population %: 9.49
Missing GDPpc %: 55.81


,Year,Sport,Country,NOC,Prev_Sport_Medals,Athlete_Count,Is_Host,Population,GDP_per_capita,log_Pop,log_GDPpc,Gold,Silver,Bronze,Total_Medals
0,2008,Taekwondo,Afghanistan,AFG,0.0,2,0,27294031.0,2191.5044,17.122179,7.692800,0,0,1,1
1,2012,Taekwondo,Afghanistan,AFG,1.0,2,0,30696958.0,2985.3190,17.239674,8.001797,0,0,1,1
2,2024,Wrestling,Albania,ALB,0.0,0,0,2947436.0,20591.4860,14.896447,9.932682,0,0,2,2
10,1984,Boxing,Algeria,ALG,0.0,7,0,21893853.0,NaN,16.901717,NaN,0,0,2,2
4,1992,Athletics,Algeria,ALG,0.0,9,0,27181094.0,NaN,17.118032,NaN,1,0,0,1


Population and GDP per capita were log-transformed to reduce skewness and model diminishing returns in economic and demographic effects on medal outcomes.

In [22]:
#8) Save cleaned tables for GitHub submission
final_model_table.to_csv(OUT_DIR / "final_model_table_summer.csv", index=False)
print("Saved:", OUT_DIR / "final_model_table_summer.csv")

Saved: outputs/final_model_table_summer.csv
